In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.nn as nn

# GPT2 using Decoder Only

## Scaled dot-product attention

Attention(Q, K, V) = softmax($\frac{QK^T}{\sqrt{d_k}}V$)

In [81]:
class FNN(nn.Module):
  def __init__(self, d_model, d_ff):
    super().__init__()
    self.linear_layer = nn.Linear(d_model, d_ff)
    self.relu = nn.ReLU()
    self.linear_layer2 = nn.Linear(d_ff, d_model)

  def forward(self, x):
    return self.linear_layer2(self.relu(self.linear_layer(x)))

# KV Cache

during inference, if model computes from scratch like just the first inference it would be very wasteful. Follwing picture illustrates this:
<img src="/content/without-kv-cache.excalidraw.png">

In [3]:
class CachedMultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, d_k=None, d_v=None):
    super().__init__()
    self.num_heads = num_heads
    self.d_k = d_k or d_model // num_heads
    self.d_v = d_v or d_model // num_heads
    self.W_q = nn.Linear(d_model, self.d_k * num_heads)
    self.W_k = nn.Linear(d_model, self.d_k * num_heads)
    self.W_v = nn.Linear(d_model, self.d_v * num_heads)
    self.W_o = nn.Linear(self.d_v * num_heads, d_model)

  def forward(self, x, cache=None):
    B, T, _ = x.shape
    q = self.W_q(x).reshape(B, T, self.num_heads, self.d_k).transpose(1, 2)
    k = self.W_k(x).reshape(B, T, self.num_heads, self.d_k).transpose(1, 2)
    v = self.W_v(x).reshape(B, T, self.num_heads, self.d_v).transpose(1, 2)

    if cache is not None:
      k = torch.cat([cache["k"], k], dim=2)
      v = torch.cat([cache["v"], v], dim=2)
    new_cache = {"k": k, "v": v}

    if cache is None:
      y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    else:
      S = k.size(2)
      mask = torch.ones(T, S, dtype=torch.bool, device=x.device).tril(diagonal=S - T)
      y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

    y = y.transpose(1, 2).reshape(B, T, self.num_heads * self.d_v)
    return self.W_o(y), new_cache

In [4]:
cmha = CachedMultiHeadAttention(768, 6)

In [5]:
randn_tensor = torch.randn(8, 1, 768)

In [6]:
output, new_cache = cmha(randn_tensor)

In [7]:
output.shape

torch.Size([8, 1, 768])

In [8]:
print(new_cache["k"].shape)
print(new_cache["v"].shape)

torch.Size([8, 6, 1, 128])
torch.Size([8, 6, 1, 128])


In [9]:
randn_tensor2 = torch.randn(8, 1, 768)

In [10]:
output2, new_cache = cmha(randn_tensor, new_cache)

In [11]:
print(output2.shape)
print(new_cache["k"].shape)
print(new_cache["v"].shape)

torch.Size([8, 1, 768])
torch.Size([8, 6, 2, 128])
torch.Size([8, 6, 2, 128])


In [12]:
class MyGPT2Block(nn.Module):
  def __init__(self, d_model, num_heads, d_ff):
    super().__init__()
    self.cmha = CachedMultiHeadAttention(d_model, num_heads)
    self.fnn = FNN(d_model, d_ff)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x, cache=None):
    old_x = x
    x_norm = self.ln1(x)
    x, new_cache = self.cmha(x_norm, cache)
    x = old_x + x
    old_x = x
    x_norm = self.ln2(x)
    x = old_x + self.fnn(x_norm)
    return x, new_cache

In [13]:
x = torch.randn(2, 8, 768)

In [14]:
my_gpt2_block = MyGPT2Block(768, 6, 768 * 2)
result, new_cache = my_gpt2_block(x)

In [15]:
print(result.shape)
print(new_cache["k"].shape)
print(new_cache["v"].shape)

torch.Size([2, 8, 768])
torch.Size([2, 6, 8, 128])
torch.Size([2, 6, 8, 128])


In [16]:
x2 = torch.randn(2, 7, 768)
result, new_cache = my_gpt2_block(x2, new_cache)

In [17]:
print(result.shape)
print(new_cache["k"].shape)
print(new_cache["v"].shape)

torch.Size([2, 7, 768])
torch.Size([2, 6, 15, 128])
torch.Size([2, 6, 15, 128])


In [18]:
class MyGPT2(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, num_layers, vocab_size, max_seq_len):
    super().__init__()
    self.embed_layer = nn.Embedding(vocab_size, d_model)
    self.pos_embed_layer = nn.Embedding(max_seq_len, d_model)
    self.layers = nn.ModuleList([
      MyGPT2Block(d_model, num_heads, d_ff) for _ in range(num_layers)
    ])
    self.ln_f = nn.LayerNorm(d_model)
    self.linear = nn.Linear(d_model, vocab_size)

  def forward(self, ids, caches=None):
      past_len = 0 if caches is None else caches[0]["k"].size(2)
      caches = caches or [None] * len(self.layers)
      B, T = ids.shape
      assert past_len + T <= self.pos_embed_layer.num_embeddings

      positions = torch.arange(past_len, past_len + T, device=ids.device).unsqueeze(0)
      x = self.embed_layer(ids) + self.pos_embed_layer(positions)

      new_caches = []
      for layer, c in zip(self.layers, caches):
          x, nc = layer(x, c)
          new_caches.append(nc)

      x = self.ln_f(x)
      return self.linear(x), new_caches

In [19]:
my_gpt2 = MyGPT2(768, 6, 768 * 2, 12, 50257, 512)

In [20]:
x1 = torch.randint(100, size=(2, 16))

In [21]:
result, new_caches = my_gpt2(x1)

In [22]:
print(result.shape)

torch.Size([2, 16, 50257])


In [23]:

print(len(new_caches))

12


In [24]:
new_caches[0]["k"].shape

torch.Size([2, 6, 16, 128])

In [25]:
x2 = torch.randint(100, size=(2, 1))
result, new_caches = my_gpt2(x2, new_caches)

In [26]:
print(result.shape)

torch.Size([2, 1, 50257])


In [27]:
print(len(new_caches))

12


In [28]:
print(new_caches[0]["k"].shape)

torch.Size([2, 6, 17, 128])


In [29]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [30]:
class TextDataset(Dataset):
  def __init__(self, token_ids, block_size):
    self.token_ids = torch.tensor(token_ids, dtype=torch.long)
    self.block_size = block_size

  def __len__(self):
    return len(self.token_ids) - self.block_size

  def  __getitem__(self, idx):
    x = self.token_ids[idx : idx + self.block_size]
    y = self.token_ids[idx + 1 : idx + self.block_size +1]
    return x, y

In [31]:
data = "Lifelong learning has become one of the most important habits a person can develop because the world is changing faster than ever before, and the knowledge or skills that are useful today may not be enough to meet the challenges of tomorrow. In the past, many people could complete their formal education, enter a profession, and expect to use roughly the same knowledge for most of their working lives, but modern technology, globalization, scientific discoveries, and social change have transformed the way people live and work. New tools appear constantly, industries evolve, and information can spread around the world within seconds, which means that individuals need to remain curious, flexible, and willing to learn throughout their lives. Lifelong learning does not only mean attending school or earning additional degrees; it includes reading books, taking online courses, learning from colleagues, practicing new skills, exploring unfamiliar subjects, asking questions, reflecting on mistakes, and gaining knowledge through daily experiences. One of the greatest benefits of lifelong learning is that it helps people adapt to change with greater confidence. When a workplace introduces new software, when a profession begins using new methods, or when a person decides to change careers, the ability to learn quickly can reduce fear and create new opportunities. A person who is accustomed to learning is more likely to see change as a challenge that can be understood rather than as a threat that must be avoided. Continuous learning can also improve career development because employers often value workers who are willing to update their skills, solve problems, communicate effectively, and take initiative. Even when a particular skill is not directly related to a person’s job, learning can strengthen abilities such as critical thinking, creativity, organization, and decision-making, which are useful in almost every profession. Beyond employment, lifelong learning can make personal life richer and more meaningful. Learning a new language can help someone communicate with people from different cultures, studying history can provide a deeper understanding of current events, learning to cook can improve independence and health, and practicing music, art, gardening, or another hobby can provide enjoyment and a sense of achievement. Learning also encourages people to remain mentally active because it requires attention, memory, reasoning, and imagination. Perhaps even more importantly, lifelong learning can teach humility by reminding people that no matter how much they know, there is always more to discover. This attitude is especially valuable in an age when people are surrounded by enormous amounts of information, some of which is inaccurate, misleading, or incomplete. A lifelong learner does not simply accept every claim but develops the habit of checking evidence, comparing sources, considering different perspectives, and changing an opinion when better information becomes available. These habits support responsible decision-making in areas such as health, finance, relationships, education, and public life. Lifelong learning can also strengthen relationships because learning often requires listening to other people and trying to understand experiences that are different from one’s own. When people approach conversations with curiosity rather than certainty, they may become more patient, respectful, and open-minded. This does not mean that they must agree with everyone, but it means they can disagree while still seeking to understand the reasons behind another person’s viewpoint. Another important aspect of lifelong learning is the ability to learn from failure. Many people are afraid of mistakes because they associate them with embarrassment or weakness, yet mistakes often provide some of the most useful lessons. A person who learns continuously can view failure as information: something did not work, so the next step is to understand why, make adjustments, and try again. This mindset is important for entrepreneurs, students, professionals, artists, athletes, and anyone trying to improve at a difficult task. However, lifelong learning requires discipline because curiosity alone is not always enough. People are busy, distractions are everywhere, and it is easy to postpone learning until there is more free time. A practical approach is to make learning a regular part of everyday life by setting realistic goals, such as reading for twenty minutes each day, completing one lesson from an online course each week, keeping notes about new ideas, or practicing a skill consistently. Small efforts may seem insignificant at first, but they can produce remarkable results when repeated over months and years. It is also helpful to choose learning goals that are meaningful rather than following every new trend. People learn more effectively when they understand why a subject matters to them and how they hope to use it. At the same time, exploring topics simply for curiosity can be valuable because unexpected connections between different fields often lead to creativity and innovation. Someone who studies both technology and psychology, for example, may understand not only how a digital product works but also how people interact with it. A person interested in business who also studies communication may become better at leadership, negotiation, and teamwork. In this way, learning across different subjects can help people develop a broader view of problems and discover solutions that might not be obvious from a single perspective. Society also benefits when people continue learning because educated, curious citizens are better prepared to understand complex issues, participate in their communities, and contribute useful ideas. Communities become stronger when people are willing to share knowledge, teach one another, and remain open to better ways of doing things. Ultimately, lifelong learning is not about knowing everything, which is impossible, but about maintaining the willingness to grow. It is a commitment to curiosity, adaptability, reflection, and improvement. The most successful learner is not necessarily the person who memorizes the most facts, but the person who knows how to ask useful questions, find reliable information, apply knowledge thoughtfully, and continue developing when circumstances change. In a world where change is unavoidable, the ability to keep learning provides a form of lasting security because even when particular technologies, jobs, or situations become outdated, the capacity to learn remains useful. By making learning a lifelong habit, people can prepare themselves for uncertainty, discover new interests, build stronger careers, make wiser decisions, connect with others, and experience the satisfaction that comes from continual growth. For these reasons, lifelong learning should not be viewed as something that ends when a person leaves school; it is an ongoing process that can shape an entire life, helping individuals remain capable, curious, and engaged no matter how much the world changes."

In [32]:
token_ids = tokenizer.encode(data)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1242 > 1024). Running this sequence through the model will result in indexing errors


In [33]:
len(token_ids)

1242

In [34]:
ds = TextDataset(token_ids, 128)

In [35]:
loader = DataLoader(ds, batch_size=8, shuffle=True)

In [36]:
optimizer = torch.optim.Adam(my_gpt2.parameters(), lr=1e-3)

In [37]:
from tqdm import tqdm

In [38]:
EPOCH_NUM = 5
device = "cuda" if torch.cuda.is_available() else "cpu"
my_gpt2.to(device)
for i in range(EPOCH_NUM):
  my_gpt2.train()
  train_loss = 0.0
  for x, y in tqdm(loader):
    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    logits, _ = my_gpt2(x)
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1)
    )
    train_loss += loss.item()
    loss.backward()
    optimizer.step()
  train_loss /= len(loader)
  print(f"Train Loss: {train_loss:.4f}")

100%|██████████| 140/140 [00:29<00:00,  4.80it/s]


Train Loss: 2.8262


100%|██████████| 140/140 [00:30<00:00,  4.66it/s]


Train Loss: 0.1060


100%|██████████| 140/140 [00:32<00:00,  4.36it/s]


Train Loss: 0.0379


100%|██████████| 140/140 [00:33<00:00,  4.20it/s]


Train Loss: 0.0302


100%|██████████| 140/140 [00:32<00:00,  4.32it/s]

Train Loss: 0.0328


In [65]:
input_ids = tokenizer.encode("Lifelong learning")
cache = None

In [74]:
logits, cache = my_gpt2(torch.tensor(input_ids).unsqueeze(0).to(device), cache)
input_ids.append(torch.argmax(logits[:, -1, :]).item())

In [75]:
input_ids

[43, 361, 21537, 4673, 468, 1716, 530, 286, 262]

In [76]:
generated_text = tokenizer.decode(input_ids)

In [77]:
generated_text

'Lifelong learning has become one of the'

In [78]:
input_ids = tokenizer.encode("It is also")
for _ in range(20):
  input_ids_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)
  logits, _ = my_gpt2(input_ids_tensor)
  next_token_logits = logits[:, -1, :]
  next_token_id = torch.argmax(next_token_logits, dim=-1)
  input_ids.append(next_token_id.item())

In [79]:
generated_text = tokenizer.decode(input_ids)

In [80]:
generated_text

'It is also helpful to choose learning goals that are meaningful rather than following every new trend. People learn more effectively when'